# LangGraph の基礎


In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="../.env", override=True)

## 単純なエージェントの実装


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch


llm = init_chat_model(
    model="gpt-5-mini",
    model_provider="openai",
    reasoning_effort="medium",
)
tools = [TavilySearch()]
llm_with_tools = llm.bind_tools(tools)


def llm_node(state: State):
    ai_message = llm_with_tools.invoke(state["messages"])
    return {"messages": [ai_message]}

In [ ]:
import json

from langchain_core.messages import ToolMessage
from langchain_core.tools import BaseTool


class BasicToolNode:
    def __init__(self, tools: list[BaseTool]) -> None:
        # {"ツール名": "ツール"} というdictを作成
        tools_by_name = {}
        for tool in tools:
            tools_by_name[tool.name] = tool
        self.tools_by_name = tools_by_name

    def __call__(self, state: State):
        latest_message = state["messages"][-1]

        tool_messages = []
        for tool_call in latest_message.tool_calls:
            tool = self.tools_by_name[tool_call["name"]]
            tool_result = tool.invoke(tool_call["args"])
            tool_messages.append(
                ToolMessage(
                    content=json.dumps(tool_result),
                    name=tool_call["name"],
                    tool_call_id=tool_call["id"],
                )
            )
        return {"messages": tool_messages}


tool_node = BasicToolNode(tools=tools)

In [ ]:
from langgraph.graph import StateGraph, START, END


graph_builder = StateGraph(State)
graph_builder.add_node("llm_node", llm_node)
graph_builder.add_node("tool_node", tool_node)


def is_last_message_tool_call(state: State):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0:
        return True
    return False


graph_builder.add_edge(START, "llm_node")
graph_builder.add_conditional_edges(
    "llm_node",
    is_last_message_tool_call,
    {
        True: "tool_node",
        False: END,
    },
)
graph_builder.add_edge("tool_node", "llm_node")
graph = graph_builder.compile()

In [ ]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
from langchain_core.messages import HumanMessage

initial_state = {"messages": [HumanMessage("こんにちは！")]}
graph.invoke(initial_state)

In [ ]:
from langchain_core.messages import HumanMessage

initial_state = {"messages": [HumanMessage("ChatGPTの最新のニュースを教えて")]}
graph.invoke(initial_state)

In [ ]:
initial_state = {"messages": [HumanMessage("ChatGPTの最新のニュースを教えて")]}

for event in graph.stream(initial_state, stream_mode="updates"):
    for value in event.values():
        latest_message = value["messages"][-1]
        latest_message.pretty_print()
